# Clean Footfall for Tableau dashboard

In [1]:
pip install matplotlib geopandas numpy scipy folium plotly

Note: you may need to restart the kernel to use updated packages.


In [2]:
#Import packages
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np

The daily footfall 2019-2024 datasets for the different Bradford locations are loaded, concatenated and cleaned.

In [ ]:
import glob
import os

#Folder with containing all areas
folder_path = r"Footfall data 2019-2024"

#Find all CSVs in all subfolders
area_folders = glob.glob(os.path.join(folder_path, '*'))

#Define file names
custom_name = ['footfall']

#Dict to store combined dataframes by file type
combined = {name: pd.DataFrame() for name in custom_name}

for area_path in area_folders:
    #Get area name
    area_name = os.path.basename(area_path)
    
    csv_files = sorted(glob.glob(os.path.join(area_path, '*.csv')))
    
    #Loop over CSVs and assign new name
    for i, file in enumerate (csv_files):
        df = pd.read_csv(file, encoding='cp1252')
        
        #Add area column
        df['region'] = area_name
        
        #Get the new names
        new_name = custom_name[i]
        
        #Concatenate into 
        combined[new_name] = pd.concat([combined[new_name], df], ignore_index=True)

In [4]:
#Extract into dataframe
df_footfall = combined['footfall']

In [5]:
df_footfall.head()

,area,Interval,Date,Total Footfall,Average Footfall,Total Visiting,Average Visiting,Total Passing through,Average Passing through,Event took place,region,Unnamed: 9,Event Name,Event name
0,"Balancing Art - Coates Street, Bradford Foyer,...",dayOfWeek,01/01/2019,844612.85,2307.69,408949.1487,1117.349636,435663.7013,1190.340364,NaN,"Balancing Acts - 1 Coates Street, Bradford, BD...",NaN,NaN,NaN
1,"Balancing Art - Coates Street, Bradford Foyer,...",dayOfWeek,07/01/2019,771831.49,2108.83,373709.4821,1021.064542,398122.0079,1087.765458,NaN,"Balancing Acts - 1 Coates Street, Bradford, BD...",NaN,NaN,NaN
2,"Balancing Art - Coates Street, Bradford Foyer,...",dayOfWeek,14/01/2019,955770.21,2611.39,462769.9113,1264.396720,493000.2987,1346.993280,NaN,"Balancing Acts - 1 Coates Street, Bradford, BD...",NaN,NaN,NaN
3,"Balancing Art - Coates Street, Bradford Foyer,...",dayOfWeek,21/01/2019,853455.73,2331.85,413230.7414,1129.047554,440224.9886,1202.802446,NaN,"Balancing Acts - 1 Coates Street, Bradford, BD...",NaN,NaN,NaN
4,"Balancing Art - Coates Street, Bradford Foyer,...",dayOfWeek,28/01/2019,839562.82,2293.89,406503.9982,1110.667879,433058.8218,1183.222121,NaN,"Balancing Acts - 1 Coates Street, Bradford, BD...",NaN,NaN,NaN


In [6]:
df_footfall['Event took place'].value_counts()

Event took place
YES    2938
Name: count, dtype: int64

In [7]:
#Only keep columns interested in
df_footfall = df_footfall[['area', 'Interval', 'Date', 'Total Footfall']]
df_footfall.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38714 entries, 0 to 38713
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   area            38714 non-null  object 
 1   Interval        38714 non-null  object 
 2   Date            38714 non-null  object 
 3   Total Footfall  38714 non-null  float64
dtypes: float64(1), object(3)
memory usage: 1.2+ MB


In [8]:
df_footfall['Interval'].unique()

array(['dayOfWeek', 'daily', 'weekly', 'monthly'], dtype=object)

In [9]:
#Keep only the daily intervals
df_footfall = df_footfall[df_footfall['Interval'] == 'daily']

#Drop the interval column as the dataset now only contains daily data
df_footfall = df_footfall.drop(columns=['Interval'])

In [10]:
df_footfall['Date'] = pd.to_datetime(df_footfall['Date'], format='mixed', dayfirst= True)
df_footfall = df_footfall.rename(columns={'Date':'datestamp'})

In [11]:
#Rename total footfall column
df_footfall = df_footfall.rename(columns={'Total Footfall': 'estimated_actual_footfall'})

In [12]:
#Check the date range for each area
data_range_per_area = (
    df_footfall
    .groupby('area')['datestamp']
    .agg(min_date='min', max_date='max')
    .reset_index()
)

print(data_range_per_area)

                                                 area   min_date   max_date
0                                BD WALL - Wayfinders 2019-01-01 2026-01-17
1                      BD WALLS: Come on in my friend 2019-01-10 2025-01-01
2                               BD Walls - The Portal 2019-01-07 2024-12-09
3                     BD Walls : Serving the district 2019-01-07 2024-12-30
4                                      BD Walls Roots 2019-01-11 2024-10-01
5                                             Baildon 2019-01-07 2024-11-12
6   Balancing Art - Coates Street, Bradford Foyer,... 2019-02-11 2024-01-27
7                                            Bradford 2019-01-01 2026-01-04
8                              Bradford - City Centre 2019-01-07 2024-12-23
9                                        Bradford BID 2019-01-01 2026-01-04
10                               Draley Street Market 2019-01-07 2024-12-17
11                                        Lister Park 2019-01-07 2024-12-23
12          

In [13]:
#Check all areas in the data
df_footfall['area'].unique()

array(['Balancing Art - Coates Street, Bradford Foyer, Coates TER, Wootton Street',
       'BD WALLS: Come on in my friend', 'Roundwood Avenue',
       'BD Walls Roots', 'BD Walls : Serving the district',
       'BD Walls - The Portal', 'BD WALL - Wayfinders', 'Bradford',
       'Bradford BID', 'Bradford - City Centre', 'Draley Street Market',
       'Lister Park', 'Baildon', 'WILD UPLANDS'], dtype=object)

The data for 2025 is missing in certain areas, thus datasets containing this needs to be integrated.

In [ ]:
import glob
import os

#Folder with containing all areas
folder_path = r"Footfall data 2025"

#Find all CSVs in all subfolders
area_folders = glob.glob(os.path.join(folder_path, '*'))

#Define file names
custom_name = ['footfall_2025']

#Dict to store combined dataframes by file type
combined = {name: pd.DataFrame() for name in custom_name}

for area_path in area_folders:
    #Get area name
    area_name = os.path.basename(area_path)
    
    csv_files = sorted(glob.glob(os.path.join(area_path, '*.csv')))
    
    #Loop over CSVs and assign new name
    for i, file in enumerate (csv_files):
        df = pd.read_csv(file, encoding='cp1252')
        
        #Add area column
        df['region'] = area_name
        
        #Get the new names
        new_name = custom_name[i]
        
        #Concatenate into 
        combined[new_name] = pd.concat([combined[new_name], df], ignore_index=True)

In [15]:
#Extract dataframe
F_2025 = combined['footfall_2025']

#Only keep columns interested in
F_2025 = F_2025[['area', 'Interval', 'Date', 'Total Footfall']]

#Keep only the daily intervals
F_2025 = F_2025[F_2025['Interval'] == 'daily']

#Drop the interval column as the dataset now only contains daily data
F_2025 = F_2025.drop(columns=['Interval'])

#Convert date column to datetime
F_2025['Date'] = pd.to_datetime(F_2025['Date'], format='mixed', dayfirst= True)
F_2025 = F_2025.rename(columns={'Date':'datestamp'})

#Rename total footfall column
F_2025 = F_2025.rename(columns={'Total Footfall': 'estimated_actual_footfall'})


#Check
F_2025.head()

,area,datestamp,estimated_actual_footfall
7,Bradford - Penistone Hill,2019-01-04,2056.59
8,Bradford - Penistone Hill,2019-01-05,4987.24
9,Bradford - Penistone Hill,2019-01-07,2493.62
10,Bradford - Penistone Hill,2019-01-11,2056.59
11,Bradford - Penistone Hill,2019-01-12,4987.24


In [16]:
#Check date ranges for each area
data_range_per_area = (
    F_2025
    .groupby('area')['datestamp']
    .agg(min_date='min', max_date='max')
    .reset_index()
)
print(data_range_per_area)

                                                 area   min_date   max_date
0       BD Walls : Root 1 Coates St, Bradford BD5 7DL 2024-12-25 2025-12-23
1                                       BD Walls RAVO 2024-12-28 2026-01-27
2             BD walls : Come on in my friend BD5 3PX 2024-12-25 2026-01-26
3                         BD walls the portal BD1 CBH 2024-12-25 2026-01-27
4   BD walls: Serving the district Morrisons, Brad... 2024-12-25 2026-01-27
5                                            Bradford 2024-12-25 2025-12-26
6                           Bradford - Penistone Hill 2019-01-04 2026-01-15
7                                Bradford city centre 2024-12-25 2026-01-27
8                              Darley's street market 2024-12-25 2026-01-27
9                                         Lister Park 2024-12-25 2026-01-27
10                    Painting the sky - Roberts part 2024-12-25 2026-01-25


In [17]:
F_2025['area'].unique()

array(['Bradford - Penistone Hill', 'Bradford',
       'BD walls : Come on in my friend BD5 3PX', 'BD Walls RAVO',
       'BD Walls : Root 1 Coates St, Bradford BD5 7DL',
       'BD walls: Serving the district Morrisons, Bradford Road, Idle BD10',
       'BD walls the portal BD1 CBH', 'Bradford city centre',
       "Darley's street market", 'Lister Park',
       'Painting the sky - Roberts part'], dtype=object)

In [18]:
df_footfall['area'].unique()

array(['Balancing Art - Coates Street, Bradford Foyer, Coates TER, Wootton Street',
       'BD WALLS: Come on in my friend', 'Roundwood Avenue',
       'BD Walls Roots', 'BD Walls : Serving the district',
       'BD Walls - The Portal', 'BD WALL - Wayfinders', 'Bradford',
       'Bradford BID', 'Bradford - City Centre', 'Draley Street Market',
       'Lister Park', 'Baildon', 'WILD UPLANDS'], dtype=object)

In [19]:
#Rename some of the area names as they don't match between the 2019-2024 and the 2025 datasets
F_2025['area'] = F_2025['area'].replace('Bradford', 'Balancing Art - Coates Street, Bradford Foyer, Coates TER, Wootton Street')
F_2025['area'] = F_2025['area'].replace('BD Walls RAVO', 'Roundwood Avenue')
F_2025['area'] = F_2025['area'].replace('BD walls: Serving the district Morrisons, Bradford Road, Idle BD10', 'BD Walls : Serving the district')
F_2025['area'] = F_2025['area'].replace('BD walls the portal BD1 CBH', 'BD Walls - The Portal')
F_2025['area'] = F_2025['area'].replace('Bradford city centre', 'Bradford - City Centre')
F_2025['area'] = F_2025['area'].replace("Darley's street market", 'Draley Street Market')


df_footfall['area'] = df_footfall['area'].replace('Bradford', 'Bradford BID')
df_footfall['area'] = df_footfall['area'].replace('WILD UPLANDS', 'Bradford - Penistone Hill')
df_footfall['area'] = df_footfall['area'].replace('BD WALLS: Come on in my friend', 'BD walls : Come on in my friend BD5 3PX')
df_footfall['area'] = df_footfall['area'].replace('BD Walls Roots', 'BD Walls : Root 1 Coates St, Bradford BD5 7DL')
df_footfall['area'] = df_footfall['area'].replace('Baildon', 'Painting the sky - Roberts part')

In [ ]:
#Load footfall data for the 'Bradford MetOffice' aka district area
footfall_Met = pd.read_csv(r"footfall-MetOffice.csv")
footfall_Met.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2470 entries, 0 to 2469
Data columns (total 8 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   datestamp                          2470 non-null   object 
 1   centre_name                        2470 non-null   object 
 2   purchasing_power_quantile          2470 non-null   int64  
 3   estimated_actual_footfall          2306 non-null   float64
 4   estimated_actual_footfall_rolling  2470 non-null   int64  
 5   indexed_signal                     0 non-null      float64
 6   indexed_signal_rolling             0 non-null      float64
 7   source                             2470 non-null   object 
dtypes: float64(3), int64(2), object(3)
memory usage: 154.5+ KB


In [21]:
#Prepare the footfall for Bradford MetOffice area
footfall_Met = footfall_Met.drop(columns=['purchasing_power_quantile', 'indexed_signal', 'indexed_signal_rolling', 'source', 'estimated_actual_footfall_rolling'])
footfall_Met = footfall_Met.rename(columns={'centre_name': 'area'})
footfall_Met.head()

,datestamp,area,estimated_actual_footfall
0,2019-01-01,Met Office - Bradford,530996.0
1,2019-01-02,Met Office - Bradford,568621.0
2,2019-01-03,Met Office - Bradford,606939.0
3,2019-01-04,Met Office - Bradford,508695.0
4,2019-01-05,Met Office - Bradford,468546.0


In [22]:
#Add the bradford district footfall (MetOffice) and 2025 datasets
#Concatenate the now 3 datasets together (have the same columns)
footfall_mix = pd.concat([df_footfall, footfall_Met, F_2025], axis=0)
#Reset index
footfall_mix = footfall_mix.reset_index(drop= True)

#Convert datestamp to datetime
footfall_mix['datestamp'] = pd.to_datetime(footfall_mix['datestamp'])
footfall_mix.head()

,area,datestamp,estimated_actual_footfall
0,"Balancing Art - Coates Street, Bradford Foyer,...",2019-02-11,2272.41
1,"Balancing Art - Coates Street, Bradford Foyer,...",2019-02-18,2272.41
2,"Balancing Art - Coates Street, Bradford Foyer,...",2019-02-25,2272.41
3,"Balancing Art - Coates Street, Bradford Foyer,...",2019-02-27,2272.41
4,"Balancing Art - Coates Street, Bradford Foyer,...",2019-03-01,18179.26


In [23]:
#Rename all areas again for clarity and consistency
footfall_mix['area'] = footfall_mix['area'].replace('Met Office - Bradford', 'Bradford - Local Authority')
footfall_mix['area'] = footfall_mix['area'].replace('Bradford BID', 'BID')
footfall_mix['area'] = footfall_mix['area'].replace('Lister Park', 'Lister Park')
footfall_mix['area'] = footfall_mix['area'].replace('BD WALL - Wayfinders', 'BD Walls : Wayfinders')
footfall_mix['area'] = footfall_mix['area'].replace('BD Walls - The Portal', 'BD Walls : The Portal')
footfall_mix['area'] = footfall_mix['area'].replace('BD walls : Come on in my friend BD5 3PX', 'BD Walls : Come on in my friend')
footfall_mix['area'] = footfall_mix['area'].replace('BD Walls : Root 1 Coates St, Bradford BD5 7DL', 'BD Walls : Roots')
footfall_mix['area'] = footfall_mix['area'].replace('Roundwood Avenue', 'BD Walls: RAVO')
footfall_mix['area'] = footfall_mix['area'].replace('Draley Street Market', 'Darley Street Market')
footfall_mix['area'] = footfall_mix['area'].replace('Painting the sky - Roberts part', 'Roberts Park')
footfall_mix['area'] = footfall_mix['area'].replace('Bradford - City Centre', 'City Centre')

In [24]:
footfall_mix['area'].value_counts()

area
BID                                                                          5122
Bradford - Penistone Hill                                                    3342
City Centre                                                                  2960
Lister Park                                                                  2960
BD Walls : Serving the district                                              2956
Darley Street Market                                                         2919
BD Walls : Come on in my friend                                              2910
BD Walls : The Portal                                                        2864
Roberts Park                                                                 2806
BD Walls : Wayfinders                                                        2574
BD Walls : Roots                                                             2515
Bradford - Local Authority                                                   2470
BD Walls: R

In [25]:
#Check for 0s
(footfall_mix['estimated_actual_footfall'] == 0).sum()


np.int64(0)

In [26]:
#Check for duplicates
print(footfall_mix.duplicated().sum())

#Take the average of duplicates per area per date
keys = ['datestamp', 'area']
footfall_mix = footfall_mix.groupby(keys).mean().reset_index()

#Check for duplicates again
print(footfall_mix.duplicated().sum())

39
0


In [27]:
#Check for NAs
print(footfall_mix['estimated_actual_footfall'].isna().value_counts())

#Drop missing data in 'estimated_actual_footfall'
footfall_mix = footfall_mix.dropna(subset=['estimated_actual_footfall'])
#Check again
print(footfall_mix['estimated_actual_footfall'].isna().value_counts())

estimated_actual_footfall
False    32119
True       164
Name: count, dtype: int64
estimated_actual_footfall
False    32119
Name: count, dtype: int64


In [28]:
#Remove 2026 data
footfall_mix = footfall_mix[(footfall_mix['datestamp'].dt.year) != 2026]

In [29]:
#Check date ranges per area again
data_range_per_area = (
    footfall_mix
    .groupby('area')['datestamp']
    .agg(min_date='min', max_date='max')
    .reset_index()
)
print(data_range_per_area)

                                                 area   min_date   max_date
0                     BD Walls : Come on in my friend 2019-01-10 2025-12-31
1                                    BD Walls : Roots 2019-01-11 2025-12-23
2                     BD Walls : Serving the district 2019-01-07 2025-12-31
3                               BD Walls : The Portal 2019-01-07 2025-12-30
4                               BD Walls : Wayfinders 2019-01-01 2025-12-31
5                                      BD Walls: RAVO 2019-01-28 2025-12-30
6                                                 BID 2019-01-01 2025-12-31
7   Balancing Art - Coates Street, Bradford Foyer,... 2019-02-11 2025-12-26
8                          Bradford - Local Authority 2019-01-01 2025-06-21
9                           Bradford - Penistone Hill 2019-01-04 2025-12-31
10                                        City Centre 2019-01-07 2025-12-31
11                               Darley Street Market 2019-01-07 2025-12-31
12          

In [30]:
#Create year, month and weekday columns
footfall_mix['year'] = footfall_mix['datestamp'].dt.year
footfall_mix['month'] = footfall_mix['datestamp'].dt.month_name()
footfall_mix['Weekday'] = footfall_mix['datestamp'].dt.day_name()
footfall_mix.head()

,datestamp,area,estimated_actual_footfall,year,month,Weekday
0,2019-01-01,BD Walls : Wayfinders,79795.860,2019,January,Tuesday
1,2019-01-01,BID,54153.485,2019,January,Tuesday
2,2019-01-01,Bradford - Local Authority,530996.000,2019,January,Tuesday
3,2019-01-02,BD Walls : Wayfinders,17168.990,2019,January,Wednesday
4,2019-01-02,BID,158891.385,2019,January,Wednesday


# Step 2: Data Quality Check (missing dates)

In [31]:
#Calculate the number of missing days per year per area

#Create object with all dates between 2019 and 2026
all_dates = (
    pd.date_range(start= '2019-01-01', end='2025-12-31', freq='D')
    .to_frame(index= False, name='datestamp'))
all_dates['year'] = all_dates['datestamp'].dt.year

missing_days_counts = (
    footfall_mix
    .groupby(['area', 'year'])
    .apply(
        lambda df: (
            pd.Index(
                all_dates.loc[all_dates['year'] == df.name[1], 'datestamp'])
            .difference(pd.Index(df['datestamp']))
            .size
    ))
    .reset_index(name='footfall_missing_days')
)
print(missing_days_counts)

                               area  year  footfall_missing_days
0   BD Walls : Come on in my friend  2019                     34
1   BD Walls : Come on in my friend  2020                      7
2   BD Walls : Come on in my friend  2021                      0
3   BD Walls : Come on in my friend  2022                      0
4   BD Walls : Come on in my friend  2023                      1
..                              ...   ...                    ...
93                     Roberts Park  2021                      0
94                     Roberts Park  2022                     22
95                     Roberts Park  2023                      5
96                     Roberts Park  2024                     50
97                     Roberts Park  2025                     41

[98 rows x 3 columns]


C:\Users\qxnq723\AppData\Local\Temp\ipykernel_29892\2990790734.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


**Note:**
From the previous table summary, it appears that certain areas have a lot of missing days in certain years. These areas are thus removed from reporting.
The areas concerned are:
* Bradford - Penistone Hill (missing around half the dates in 2019, 2022, 2023, 2024 and 2025)
* BD Walls: RAVO (missing around 70% of days in 2024 and 2025)
* Balancing Art - Coates Street, Bradford Foyer, Coates TER, Wootton Street (missing 90% of 2024 and missing a lot of days over all years)

In [32]:
footfall_mix = footfall_mix[~footfall_mix['area'].isin(['Bradford - Penistone Hill',
                                                        'BD Walls: RAVO',
                                                        'Balancing Art - Coates Street, Bradford Foyer, Coates TER, Wootton Street'])]

# Step 3: Define time periods

In [33]:
# Define each period
period_bins = [
    pd.Timestamp('2018-12-31'),
    pd.Timestamp('2020-03-23'),
    pd.Timestamp('2022-02-24'),
    pd.Timestamp('2025-01-10'),
    pd.Timestamp('2026-01-01')]

period_labels = ['pre_covid', 'covid', 'post_covid', 'BCoC_events']

#Create a column with periods in dataset
footfall_mix['Period'] = pd.cut(
    footfall_mix['datestamp'],
    bins = period_bins,
    labels= period_labels,
    right= True)

#Check
footfall_mix.tail()

,datestamp,area,estimated_actual_footfall,year,month,Weekday,Period
32092,2025-12-31,BID,206622.525,2025,December,Wednesday,BCoC_events
32094,2025-12-31,City Centre,176594.750,2025,December,Wednesday,BCoC_events
32095,2025-12-31,Darley Street Market,8165.790,2025,December,Wednesday,BCoC_events
32096,2025-12-31,Lister Park,55627.090,2025,December,Wednesday,BCoC_events
32097,2025-12-31,Roberts Park,1891.240,2025,December,Wednesday,BCoC_events


# Step 4: Add geometries

In [ ]:
#Load file with different geographical areas of Bradford
GDF_areas = gpd.read_file(r"areas.geojson")
GDF_areas.head(10)

,centre_name,geometry
0,Bowling Park - BD4 7,"POLYGON ((-1.74342 53.78017, -1.74338 53.78022..."
1,Wibsey Park - BD6 3,"POLYGON ((-1.79115 53.76468, -1.79115 53.76482..."
2,Cliffe Castle Park - BD20 6,"POLYGON ((-1.91566 53.87713, -1.91694 53.87755..."
3,Lister Park - BD9 4,"POLYGON ((-1.77676 53.81297, -1.77673 53.81299..."
4,Bradford BID,"POLYGON ((-1.75387 53.78933, -1.75416 53.78928..."
5,Met Office - Bradford,"POLYGON ((-1.88042 53.96298, -1.87951 53.96284..."
6,Bradford - City Centre,"POLYGON ((-1.75155 53.79246, -1.75298 53.79139..."
7,Bradford - Penistone Hill,"POLYGON ((-1.97655 53.82517, -1.97316 53.8279,..."


In [35]:
GDF_areas = GDF_areas.rename(columns={'centre_name': 'area'})
GDF_areas['area'] = GDF_areas['area'].replace('Lister Park - BD9 4', 'Lister Park')
GDF_areas['area'] = GDF_areas['area'].replace('Bradford BID', 'BID')
GDF_areas['area'] = GDF_areas['area'].replace('Met Office - Bradford', 'Bradford - Local Authority')
GDF_areas['area'] = GDF_areas['area'].replace('Bradford - City Centre', 'City Centre')

**Important Note:** Here I create point geometries for the BD Walls and Darley Street Market locations, where footfall data was collected. However, these 'Point' geometries only for visualization purposes. In reality, the footfall data was collected for areas surrounding these points (customized geometries), but I have unfortunately not been provided the geometries for these. This part is mainly to create a visualization for a presentation slide.


**Create point geometries for the BD walls and Darley Street Market:**

The adress/location of the different BD Walls was collected from the [Bradford 2025 website](https://bradford2025.co.uk/programme/bd-walls/).

* 'BD Walls : Wayfinders': 15 Park Rd, Bingley BD16 4BD, UK
* 'BD Walls : Serving the district': 275 Bradford Rd, Idle, Bradford BD10 8EG, UK
* 'BD Walls : The Portal': 8 Church Bank, Bradford BD1 4DZ, UK
* 'BD Walls : Come on in my friend' : Reevy Rd, Bradford BD6 3PU, UK
* 'BD Walls : Roots': 1 Coates St, Bradford BD5 7DL, UK
* 'BD Walls: RAVO': Roundwood Ave, Bradford BD10 0LE, UK
* 'Balancing Art - Coates Street, Bradford Foyer, Coates TER, Wootton Street'
* 'Darley Street Market'

In [36]:
#Create geometry points to locate the BD walls and cultural venues
#These were manually scraped from the Bradford 2025 website and internet

point_coordinates = {
    'BD Walls : Wayfinders' : (53.8494997,-1.8396872),
    'BD Walls : The Portal' : (53.7953434,-1.7494453),
    'BD Walls : Serving the district' : (53.8249981,-1.7367139),
    'BD Walls : Come on in my friend': (53.7647511, -1.7903011),
    'BD Walls : Roots' : (53.7797565,-1.7605108),
    'BD Walls: RAVO' : (53.8178064,-1.7108643),
    'Darley Street Market' : (53.7955833,-1.7724355)
}

#Transform to dataframe
df_points = pd.DataFrame.from_dict(
    point_coordinates,
    orient='index',
    columns=['lat', 'lon']
).reset_index().rename(columns={'index': 'area'})

#Check
df_points.head()

,area,lat,lon
0,BD Walls : Wayfinders,53.849500,-1.839687
1,BD Walls : The Portal,53.795343,-1.749445
2,BD Walls : Serving the district,53.824998,-1.736714
3,BD Walls : Come on in my friend,53.764751,-1.790301
4,BD Walls : Roots,53.779756,-1.760511


In [37]:
#Create geometries from coordinates
df_points= gpd.GeoDataFrame(df_points, 
                           geometry=gpd.points_from_xy(df_points['lon'], df_points['lat']),
                           crs='EPSG:4326')

#Check
df_points.head()


,area,lat,lon,geometry
0,BD Walls : Wayfinders,53.849500,-1.839687,POINT (-1.83969 53.8495)
1,BD Walls : The Portal,53.795343,-1.749445,POINT (-1.74945 53.79534)
2,BD Walls : Serving the district,53.824998,-1.736714,POINT (-1.73671 53.825)
3,BD Walls : Come on in my friend,53.764751,-1.790301,POINT (-1.7903 53.76475)
4,BD Walls : Roots,53.779756,-1.760511,POINT (-1.76051 53.77976)


In [38]:
#drop the lat lon columns
df_points = df_points.drop(columns={'lat', 'lon'})

In [39]:
#Concatenate the different polygons and points there is footfall data for
Bradford_poly_pts = pd.concat([GDF_areas, df_points], axis=0)
Bradford_poly_pts.head(20)

,area,geometry
0,Bowling Park - BD4 7,"POLYGON ((-1.74342 53.78017, -1.74338 53.78022..."
1,Wibsey Park - BD6 3,"POLYGON ((-1.79115 53.76468, -1.79115 53.76482..."
2,Cliffe Castle Park - BD20 6,"POLYGON ((-1.91566 53.87713, -1.91694 53.87755..."
3,Lister Park,"POLYGON ((-1.77676 53.81297, -1.77673 53.81299..."
4,BID,"POLYGON ((-1.75387 53.78933, -1.75416 53.78928..."
5,Bradford - Local Authority,"POLYGON ((-1.88042 53.96298, -1.87951 53.96284..."
6,City Centre,"POLYGON ((-1.75155 53.79246, -1.75298 53.79139..."
7,Bradford - Penistone Hill,"POLYGON ((-1.97655 53.82517, -1.97316 53.8279,..."
0,BD Walls : Wayfinders,POINT (-1.83969 53.8495)
1,BD Walls : The Portal,POINT (-1.74945 53.79534)


I scrap the Roberts Park geometry as it was not provided. The green spaces file was dowloaded from the [Ordnance Survey](https://osdatahub.os.uk/data/downloads/open/OpenGreenspace). From the green spaces shapefile, the Roberts Park polygon is collected.

In [ ]:
#Load file with different green spaces from Ordnance Survey
Green_spaces = gpd.read_file(r"SE_GreenspaceSite.shp")
Green_spaces.head()

,id,function,distName1,distName2,distName3,distName4,geometry
0,3D6C4AE3-840B-80C3-E063-8ECAA00A8EB3,Playing Field,None,None,None,None,"POLYGON Z ((415736.67 432338.02 0, 415750.01 4..."
1,3D6C4AE4-30C4-80C3-E063-8ECAA00A8EB3,Play Space,Hope Park,None,None,None,"POLYGON Z ((415670.18 430406.08 0, 415661.4 43..."
2,3D6C4ACC-1106-80C3-E063-8ECAA00A8EB3,Public Park Or Garden,None,None,None,None,"POLYGON Z ((415812.26 431549.17 0, 415800.98 4..."
3,3D6C4B27-670E-80C3-E063-8ECAA00A8EB3,Religious Grounds,All Saints' Church,None,None,None,"POLYGON Z ((415749.64 432057.02 0, 415712 4320..."
4,3D6C4B27-6C15-80C3-E063-8ECAA00A8EB3,Play Space,None,None,None,None,"POLYGON Z ((415727.72 431565.95 0, 415734.2 43..."


In [41]:
#Search for polygon containing Roberts Park
Green_spaces[Green_spaces['distName1'].str.contains('Roberts Park', na=False, case=False)]

,id,function,distName1,distName2,distName3,distName4,geometry
366,3D6C4B24-2FFB-80C3-E063-8ECAA00A8EB3,Public Park Or Garden,Roberts Park,None,None,None,"POLYGON Z ((413968.48 438278.07 0, 413972.6 43..."


In [42]:
#Isolate that polygon
RobertsPark = Green_spaces[Green_spaces['distName1'] == 'Roberts Park']

In [43]:
#Check CRS
print(RobertsPark.crs)

#Convert the CRS 
RobertsPark = RobertsPark.to_crs(epsg= 4326)

#Check CRS
print(RobertsPark.crs)

EPSG:27700
EPSG:4326


In [44]:
#Keep only necessary columns
RobertsPark = RobertsPark[['distName1', 'geometry']]
#Rename column
RobertsPark = RobertsPark.rename(columns={'distName1': 'area'})


#Add the polygon to the geometries dataset
Bradford_Geo = pd.concat([Bradford_poly_pts, RobertsPark], axis=0)
Bradford_Geo.head(20)

,area,geometry
0,Bowling Park - BD4 7,"POLYGON ((-1.74342 53.78017, -1.74338 53.78022..."
1,Wibsey Park - BD6 3,"POLYGON ((-1.79115 53.76468, -1.79115 53.76482..."
2,Cliffe Castle Park - BD20 6,"POLYGON ((-1.91566 53.87713, -1.91694 53.87755..."
3,Lister Park,"POLYGON ((-1.77676 53.81297, -1.77673 53.81299..."
4,BID,"POLYGON ((-1.75387 53.78933, -1.75416 53.78928..."
5,Bradford - Local Authority,"POLYGON ((-1.88042 53.96298, -1.87951 53.96284..."
6,City Centre,"POLYGON ((-1.75155 53.79246, -1.75298 53.79139..."
7,Bradford - Penistone Hill,"POLYGON ((-1.97655 53.82517, -1.97316 53.8279,..."
0,BD Walls : Wayfinders,POINT (-1.83969 53.8495)
1,BD Walls : The Portal,POINT (-1.74945 53.79534)


In [45]:
#Merge the polygons and the footfall data
DF = footfall_mix.merge(Bradford_Geo[['area', 'geometry']], on='area')
DF = gpd.GeoDataFrame(DF, geometry='geometry')
DF = DF.to_crs(epsg=4326)
#Check CRS
print(DF.crs)

EPSG:4326


In [46]:
DF.head(30)

,datestamp,area,estimated_actual_footfall,year,month,Weekday,Period,geometry
0,2019-01-01,BD Walls : Wayfinders,79795.860,2019,January,Tuesday,pre_covid,POINT (-1.83969 53.8495)
1,2019-01-01,BID,54153.485,2019,January,Tuesday,pre_covid,"POLYGON ((-1.75387 53.78933, -1.75416 53.78928..."
2,2019-01-01,Bradford - Local Authority,530996.000,2019,January,Tuesday,pre_covid,"POLYGON ((-1.88042 53.96298, -1.87951 53.96284..."
3,2019-01-02,BD Walls : Wayfinders,17168.990,2019,January,Wednesday,pre_covid,POINT (-1.83969 53.8495)
4,2019-01-02,BID,158891.385,2019,January,Wednesday,pre_covid,"POLYGON ((-1.75387 53.78933, -1.75416 53.78928..."
5,2019-01-02,Bradford - Local Authority,568621.000,2019,January,Wednesday,pre_covid,"POLYGON ((-1.88042 53.96298, -1.87951 53.96284..."
6,2019-01-03,BD Walls : Wayfinders,20040.120,2019,January,Thursday,pre_covid,POINT (-1.83969 53.8495)
7,2019-01-03,BID,56947.585,2019,January,Thursday,pre_covid,"POLYGON ((-1.75387 53.78933, -1.75416 53.78928..."
8,2019-01-03,Bradford - Local Authority,606939.000,2019,January,Thursday,pre_covid,"POLYGON ((-1.88042 53.96298, -1.87951 53.96284..."
9,2019-01-04,BD Walls : Wayfinders,20810.900,2019,January,Friday,pre_covid,POINT (-1.83969 53.8495)


In [48]:
#Save the cleaned dataset
DF.to_csv('footfall_Tableau_Clean.csv')

In [49]:
#Create GeoJSON file
DF.to_file('footfall_Tableau_Clean.geojson', driver='GeoJSON')

In [50]:
DF['area'].unique()

array(['BD Walls : Wayfinders', 'BID', 'Bradford - Local Authority',
       'BD Walls : Serving the district', 'BD Walls : The Portal',
       'City Centre', 'Darley Street Market', 'Lister Park',
       'Roberts Park', 'BD Walls : Come on in my friend',
       'BD Walls : Roots'], dtype=object)